## Preliminares

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import PowerTransformer
import sklearn.impute as skl_imp
from sklearn.experimental import enable_iterative_imputer # Necesario para usar skl_imp, no borrar
from src.config import data_folder
from src.transform import *

In [2]:
# Abrir archivo raw_data
df = pd.read_parquet(f"{data_folder}/raw_data.parquet")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11391 entries, 0 to 11390
Data columns (total 28 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   Date                         11391 non-null  datetime64[ns]
 1   Ticker                       11391 non-null  object        
 2   Close                        11391 non-null  float64       
 3   Open                         11391 non-null  float64       
 4   Volume                       11391 non-null  float64       
 5   DateAdded                    6705 non-null   object        
 6   Sector                       11391 non-null  object        
 7   Industry                     11391 non-null  object        
 8   TotalRevenue                 11391 non-null  float64       
 9   GrossProfit                  10523 non-null  float64       
 10  OperatingIncome              11391 non-null  float64       
 11  NetIncome                    11391 non-nu

In [ ]:
# Para mejor visualización, se expresan las columnas financieras y volumen en miles
df = financieras_en_millones(df)
df['Volume'] = df['Volume']/10**6

In [4]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
Date,11391,2023-08-23 16:59:09.749802240,2020-09-01 00:00:00,2022-03-01 00:00:00,2023-09-01 00:00:00,2025-03-01 00:00:00,2026-06-01 00:00:00,NaN
Close,11391.0,1038.957576,0.57,41.111301,82.443237,164.889503,775000.0,22846.03821
Open,11391.0,1006.586258,0.53,40.56363,81.300003,161.215086,775648.0,22096.142694
Volume,11391.0,328636.592088,0.2,44149.05,103240.1,257951.7,36280458.0,1250868.955568
TotalRevenue,11391.0,6587181.907667,-12134000.0,1277100.5,2389000.0,5259000.0,227173000.0,14888047.39344
GrossProfit,10523.0,2361543.95559,-24353000.0,422542.0,832200.0,1828000.0,137192000.0,6374194.846293
OperatingIncome,11391.0,966680.232928,-55512000.0,125685.0,304000.0,820000.0,77648000.0,2942496.474229
NetIncome,11391.0,717643.038538,-43621000.0,65504.5,199200.0,580000.0,62578000.0,2669958.899471
EBITDA,11391.0,1143504.68682,-55512000.0,155000.0,370333.0,978000.0,84427000.0,3366033.895861
BasicAverageShares,11345.0,641186.144313,0.21,95479.927,219670.0,550277.0,30864000.0,1742340.601937


## Corrección de errores

In [5]:
# Definir las columnas a visualizar
columnas_clave = ['Ticker', 'Date', 'TotalDebt', 'CurrentDebt', 'LongTermDebt', 'TotalDebt']

# Usar .loc[condicion_de_filas, lista_de_columnas]
anomalias = df.loc[df['TotalDebt'] < 0, columnas_clave] # No pueden existir valores negativos de deuda

print(anomalias)

      Ticker       Date   TotalDebt  CurrentDebt  LongTermDebt   TotalDebt
9468    STLD 2024-03-01  -2826550.0     459987.0    -3286537.0  -2826550.0
9469    STLD 2024-06-01  -3144332.0     425696.0    -3570028.0  -3144332.0
9471    STLD 2024-12-01  -3115335.0     882013.0    -3997348.0  -3115335.0
10252    TXN 2025-03-01 -27299000.0     750000.0   -28049000.0 -27299000.0


In [ ]:
"""
Se observa en los 4 casos que tomando los valores absolutos, 
se cumple la igualdad: TotalDebt = CurrentDebt + LongTermDebt.
Se consideran entonces errores de signo al ingresar los datos.
"""

# Identificar las filas donde LongTermDebt es negativo
filtro_deuda = df['LongTermDebt'] < 0

# Se corrigen usando valor absoluto
df.loc[filtro_deuda, 'LongTermDebt'] = df['LongTermDebt'].abs()

# Recalcular la Deuda Total
df.loc[filtro_deuda, 'TotalDebt'] = df['CurrentDebt'] + df['LongTermDebt']

In [ ]:
# Verificar cambios
df.iloc[9468][columnas_clave]

In [ ]:
df.iloc[9469][columnas_clave]

In [ ]:
df.iloc[9471][columnas_clave]

In [ ]:
df.iloc[10252][columnas_clave]

In [ ]:
anomalias = df.loc[df['CurrentDebt'] < 0, columnas_clave]
print(anomalias)

In [ ]:
# Se imputan parte de los NaNs en variables de Deuda antes de calcular métricas, 
# mediante las relaciones contables entre ellas.
df_debt_imputed = imputar_deuda(df)
mostrar_missings(df_debt_imputed)

In [ ]:
#  Pasar DateAdded a formato datetime, los NaN se vuelven NaT (not a time)
df_debt_imputed['DateAdded'] = pd.to_datetime(df_debt_imputed['DateAdded'], errors='coerce')
# Convertir a YearsSinceAdded, aqui los nulos regresan a NaN
df_debt_imputed['YearsSinceAdded'] = round(((pd.Timestamp.now() - df_debt_imputed['DateAdded']).dt.days / 365.25), 0)

# 3. Se asignan a cero años los tickers que no pertenecen al Índice S&P 500
df_debt_imputed['YearsSinceAdded'] = df_debt_imputed['YearsSinceAdded'].fillna(0)

# Eliminar la columna original
df_debt_imputed.drop('DateAdded', axis=1, inplace=True)
mostrar_missings(df_debt_imputed)

In [ ]:
# Se aplica forward fill para cubrir posibles huecos de hasta 1 período
df_debt_imputed = cubrir_huecos(df_debt_imputed,limite=1)
mostrar_missings(df_debt_imputed)

In [ ]:
# Se convierten las variables de flujo trimestrales a valores TTM (ventana móvil de 4 trimestres)
df_debt_imputed = transformar_flujos_a_ttm(df_debt_imputed)
mostrar_missings(df_debt_imputed)

In [ ]:
# Calcular métricas
df_with_metrics, crecimiento_cols = calcular_metricas(df_debt_imputed)
mostrar_missings(df_with_metrics)

In [ ]:
# Se aplica imputación transversal para las columnas de crecimiento
df_with_metrics = imputar_transversal(df_with_metrics, crecimiento_cols)
mostrar_missings(df_with_metrics)

In [ ]:
# Calcular los retornos trimestrales, varianza del activo y covarianza con el mercado para cada ticker
# Se abre el fichero de precios del Índice del Mercado
df_index = pd.read_parquet(f"{data_folder}/market_index.parquet")
df_with_features = calcular_retornos(df_with_metrics, df_index)
mostrar_missings(df_with_features)

## Missing Values

In [ ]:
# Se separan columnas numéricas y no numéricas
df_cont = df_with_features.select_dtypes(include='number')
df_non_numeric = df_with_features.select_dtypes(exclude='number')

# Comprobar valores infinitos antes de imputación multivariable
print(np.isinf(df_cont).sum())

In [ ]:
# NaN Restantes: Imputación multivariable con IterativeImputer sobre numéricas
# Imputador: Chain equations
imputer_itImp = skl_imp.IterativeImputer(max_iter=10, random_state=0)

df_cont_imputed = pd.DataFrame(imputer_itImp.fit_transform(df_cont),columns=df_cont.columns)
mostrar_missings(df_cont_imputed)

In [ ]:
# Se vuelven a unir las columnas numéricas y no numéricas
df_imputed = pd.concat([df_cont_imputed, df_non_numeric], axis=1)

## Transformaciones

In [ ]:
# Se calculan tamaños relativos: RelativeAssets y RelativeRevenue
df_transformed = calcular_relative_size(df_imputed)
df_transformed.info()

In [ ]:
# Se expresan columnas de capitalización y de mercado en billions
cols = [
    'MarketCap', 
    'EnterpriseValue', 
    'TotalMarketAssets', 
    'TotalMarketRevenue'
    ]

for col in cols:
    df_transformed[col] = df_transformed[col] / 10**9

In [ ]:
# Convertir Sector y Industry a tipo category
df_transformed['Sector'] = df_transformed['Sector'].astype('category')
df_transformed['Industry'] = df_transformed['Industry'].astype('category')

# Valores unicos en Sector
df_transformed['Sector'].value_counts()

In [ ]:
# Valores unicos en Industry
df_transformed['Industry'].value_counts()

In [ ]:
# Distribución de variables contínuas
df_transformed.describe().round(4).T

In [ ]:
# Coeficientes de asimetría
df_transformed.select_dtypes(include="number").skew().sort_values(ascending=False)

In [ ]:
# Graficar
columna_a_graficar = 'NetDebtToEbitda' # indicar columna para el grafico
plot(df_transformed[columna_a_graficar])

In [ ]:
# Transformaciones logarítmicas

columnas_a_transformar = [ 
    'RelativeAssets',
    'RelativeRevenue'
    ]
for columna in columnas_a_transformar:
    df_transformed[columna] = df_transformed[columna].fillna(0)
    df_transformed[f'{columna}_log'] = np.log1p(df_transformed[columna])
    df_transformed.drop(columna, axis=1, inplace=True)

# Coeficientes de asimetria actualizado
df_transformed.select_dtypes(include="number").skew().sort_values(ascending=False)

In [ ]:
# Definir columnas que saltean la "winsorización"
columnas_intactas = cols_monetarias + [
    # Variables de precio (posibles label)
    'Close',
    'Open',    
    'TrailingPE',
    'EnterpriseToEbitda',
    'PriceToBook',
    # Otras
    'Date', 
    'Ticker'       
    ]

# Separar el dataset
df_passthrough = df_transformed[columnas_intactas].copy()
df_transformed_features = df_transformed.drop(columns=columnas_intactas)

## Gestión de Outliers

Se winsorizan los valores atipicos en las variables continuas que cumplan los siguientes criterios:

Para variables simetricas:
* A mas de 3 desviaciones tipicas de la media.
* Mas de 3 rangos intercuartilicos.

Para variables asimetricas (modulo del coeficiente de asimetrica mayor a 1):
* A mas de 3 MADs de la mediana.
* Mas de 3 rangos intercuartilicos.

In [ ]:
# Outliers
df_cont_transformed = df_transformed_features.select_dtypes(include="number")
df_winsor = df_cont_transformed.apply(lambda x: gestiona_outliers(x, clas='winsor'))

In [ ]:
# Coeficientes de asimetria luego de winsorizar
df_winsor.skew().sort_values(ascending=False)

In [ ]:
# Visualizar cambios
columna_a_graficar = 'RelativeAssets_log' # indicar columna para el grafico
plot(df_winsor[columna_a_graficar])

In [ ]:
df_winsor.describe().T

## Concatenación final de columnas

In [ ]:
df_non_numeric_transformed = df_transformed_features.select_dtypes(exclude='number')
# Se unen variables contínuas transformadas y variables no numéricas
df_combined = pd.concat([df_non_numeric_transformed, df_winsor], axis=1)

# Unir con las columnas que fueron salteadas
df_final = pd.concat([df_passthrough, df_combined], axis=1)
df_final.info()

In [ ]:
# Guardar datos extraidos en fichero clean_data
df_final.to_parquet(f"{data_folder}/clean_data.parquet")